In [2]:
import argparse
import torch

from tqdm import tqdm
import pickle

from scipy.stats import mannwhitneyu

In [87]:
all_acts = torch.load("data/summarized_acts_top_q.pt")
all_properties = torch.load("metadata/ptn_fam_tensor_nonzero.pt")

print(all_acts.shape)

/scratch/login/ipykernel_1186237/3538289224.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  all_acts = torch.load("data/summarized_acts_top_q.pt")
/scratch/login/ipykern

torch.Size([10000, 28672])


In [88]:
with open("metadata/list_of_desired_latents_2pkea.pkl", 'rb') as f:
    subset_list = list(set(pickle.load(f)))
    subset_list.sort()

In [89]:
all_acts = all_acts[:,subset_list]

In [90]:

latents_property_pvals = torch.zeros(all_acts.shape[1], all_properties.shape[1])
latents_property_statistic = torch.zeros(all_acts.shape[1], all_properties.shape[1])
latents_property_effect_size = torch.zeros(size=(all_acts.shape[1], all_properties.shape[1],4))

In [91]:
len(subset_list)

188

In [92]:
latents_property_pvals.shape

torch.Size([188, 10096])

In [93]:
all_acts.shape

torch.Size([10000, 188])

In [94]:
all_properties.shape

torch.Size([10000, 10096])

In [95]:
import json

with open("/work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/results/layer_latent_dicts/layer_latent_dict_2PKEA_0.70.json", "r") as f:
    pkea_latents = json.load(f)

In [96]:
offsets = {k: i*4096 for i,k in enumerate(pkea_latents.keys())}
full_indices_pkea = [j+offsets[layer_id] for layer_id in pkea_latents.keys() for j in pkea_latents[layer_id]]
latent_ids_both_proteins = list(set(full_indices_pkea))
latent_ids_both_proteins.sort()

In [97]:
subset_list == latent_ids_both_proteins

True

In [98]:
len(latent_ids_both_proteins)

188

In [99]:
def to_flattened_id(layer_id: int, lat_id: int) -> int:
    if layer_id % 4 != 0:
        raise ValueError("layer_id must be a multiple of 4 (e.g., 4, 8, 12, ...)")
    if not (0 <= lat_id < 4096):
        raise ValueError("lat_id must be in [0, 4096)")
    return ((layer_id // 4) - 1) * 4096 + lat_id

In [ ]:
test_j = [9339, 8906, 9029]
layer_i = 16
latent_ind = 2646
flattened_id = to_flattened_id(int(layer_i), int(latent_ind))
latent_i = subset_list.index(flattened_id)
print(latent_i)


83


In [102]:
import pandas as pd
interpro_annotations_nonzero = pd.read_csv("metadata/interpro_entry_list_mapping_nonzero.csv")
interpro_annotations_nonzero.iloc[9339]["ENTRY_NAME"]

'HAD-like superfamily'

In [ ]:
12:2112, 9059

In [103]:
i = latent_i
for j in test_j:
    tmp_x = all_acts[all_properties[:,j] != 0,i]
    tmp_y = all_acts[all_properties[:,j] == 0,i]
    latents_property_pvals[latent_i,j] = mannwhitneyu(tmp_x, tmp_y, alternative="greater").pvalue
    latents_property_statistic[i,j] = mannwhitneyu(tmp_x, tmp_y, alternative="greater").statistic
    latents_property_effect_size[i,j,0] = tmp_x.mean()
    latents_property_effect_size[i,j,1] = tmp_y.mean()
    latents_property_effect_size[i,j,2] = all_acts[:,0].std()
    latents_property_effect_size[i,j,3] = (tmp_x.mean() - tmp_y.mean())/(all_acts[:,0].std())
    print(f"  p-value: {latents_property_pvals[i,j]:.3e}")
    print(f"  Mann-Whitney U statistic: {latents_property_statistic[i,j]:.1f}")
    print(f"  Mean for positive class: {latents_property_effect_size[i,j,0]:.3f}")
    print(f"  Mean for negative class: {latents_property_effect_size[i,j,1]:.3f}")
    print(f"  Standard deviation: {latents_property_effect_size[i,j,2]:.3f}")
    print(f"  Effect size (Cohen's d): {latents_property_effect_size[i,j,3]:.3f}")

  p-value: 9.970e-01
  Mann-Whitney U statistic: 76666.0
  Mean for positive class: 16.824
  Mean for negative class: 17.522
  Standard deviation: 0.634
  Effect size (Cohen's d): -1.101
  p-value: 9.952e-01
  Mann-Whitney U statistic: 83083.0
  Mean for positive class: 16.870
  Mean for negative class: 17.522
  Standard deviation: 0.634
  Effect size (Cohen's d): -1.029
  p-value: 1.000e+00
  Mann-Whitney U statistic: 18880.5
  Mean for positive class: 15.906
  Mean for negative class: 17.523
  Standard deviation: 0.634
  Effect size (Cohen's d): -2.551


In [65]:
import numpy as np
from scipy.stats import mannwhitneyu

try:
    import torch
except Exception:
    torch = None

def to_numpy_1d(a):
    # Convert numpy/torch/list to 1D numpy and drop NaNs/Infs
    if torch is not None and isinstance(a, torch.Tensor):
        a = a.detach().cpu().numpy()
    else:
        a = np.asarray(a)
    a = np.ravel(a)
    if a.size == 0:
        return a
    mask = np.isfinite(a)
    return a[mask]

i = latent_i  # latent index
# for i in tqdm(range(latents_property_pvals.shape[0])):
for j in test_j[:1]:
    # Define classes:
    # x = "positives" where property != 0
    # y = "negatives" where property == 0
    # If your gold standard uses the opposite, AUROC < 0.5 will reveal it.
    x_raw = all_acts[all_properties[:, j] != 0, i]
    y_raw = all_acts[all_properties[:, j] == 0, i]

    x = to_numpy_1d(x_raw)
    y = to_numpy_1d(y_raw)

    n_pos, n_neg = x.size, y.size
    if n_pos == 0 or n_neg == 0:
        print(f"Property j={j}: one of the groups is empty (n_pos={n_pos}, n_neg={n_neg}); skipping.\n")
        continue

    # Mann–Whitney U tests
    res_greater = mannwhitneyu(x, y, alternative="greater")
    res_two = mannwhitneyu(x, y, alternative="two-sided")

    U = float(res_greater.statistic)
    auc = U / (n_pos * n_neg)  # AUROC = P(X > Y) + 0.5 * P(X == Y) under mid-ranks

    mean_pos = float(np.mean(x))
    mean_neg = float(np.mean(y))

    if n_pos + n_neg > 2:
        pooled_std = float(np.std(np.concatenate([x, y]), ddof=1))
    else:
        pooled_std = np.nan

    cohens_d = (mean_pos - mean_neg) / pooled_std if np.isfinite(pooled_std) and pooled_std > 0 else np.nan

    if auc < 0.2:
        print(f"  AUROC < 0.2; swapping classes would give AUROC={1-auc:.3f}")
    if auc > 0.95:
        cur_lat_id = subset_list[i]  # not subset_list[latent_i]
        layer_id = ((cur_lat_id // 4096) + 1) * 4
        lat_id = int(cur_lat_id % 4096)
        print(f"  AUROC > 0.8; latent {i} is in layer {layer_id} with latent ID {lat_id}")
        print(f"Property latent={i}, j={j}")
        print(f"  n_pos={n_pos}, n_neg={n_neg}")
        print(f"  Mean(pos)={mean_pos:.3f}, Mean(neg)={mean_neg:.3f}")
        if np.isfinite(pooled_std):
            print(f"  Pooled std={pooled_std:.3f}, Cohen's d={cohens_d:.3f}")
        else:
            print("  Pooled std=nan, Cohen's d=nan")
        print(f"  Mann-Whitney U={U:.1f}")
        print(f"  p-value (greater)={res_greater.pvalue:.3e}")
        print(f"  p-value (two-sided)={res_two.pvalue:.3e}")
        print(f"  AUROC={auc:.3f}")
    # print()

  AUROC > 0.8; latent 80 is in layer 12 with latent ID 2112
Property latent=80, j=9059
  n_pos=63, n_neg=9937
  Mean(pos)=11.973, Mean(neg)=0.204
  Pooled std=1.090, Cohen's d=10.797
  Mann-Whitney U=626031.0
  p-value (greater)=8.754e-85
  p-value (two-sided)=1.751e-84
  AUROC=1.000


In [ ]:
for i in tqdm(range(latents_property_pvals.shape[0])):
    for j in range(latents_property_pvals.shape[1]):
        tmp_x = all_acts[all_properties[:,j] != 0,i]
        tmp_y = all_acts[all_properties[:,j] == 0,i]
        latents_property_pvals[i,j] = mannwhitneyu(tmp_x, tmp_y, alternative="greater").pvalue
        latents_property_statistic[i,j] = mannwhitneyu(tmp_x, tmp_y, alternative="greater").statistic
        latents_property_effect_size[i,j,0] = tmp_x.mean()
        latents_property_effect_size[i,j,1] = tmp_y.mean()
        latents_property_effect_size[i,j,2] = all_acts[:,0].std()
        latents_property_effect_size[i,j,3] = (tmp_x.mean() - tmp_y.mean())/(all_acts[:,0].std())